In [1]:
from glob import glob
from IPython.display import display

import polars as pl
import torch

from fmlib.data.dataset import TabularDataset
from fmlib.data.dataset.collate_fn import TabularCollateFn, UpliftCollateFn

## Data sample

In [2]:
train_path = "data/campaign_demo/cc_processed/train"

first_parquet_file = glob(train_path + "/*.parquet")[0]
data_sample = pl.read_parquet(first_parquet_file)
display(data_sample.head())

epk_id,month_part,report_month,bucket_num,target_attr_1,target_attr_2,target_attr_3,cat_features,num_features,seq_hidden_state
i64,date,date,i32,i32,i32,i32,list[i64],list[f32],list[f64]
1126087785893106041,2024-03-31,2024-05-31,323,1,2,1,"[3, 5, … 171]","[-0.295914, 0.216067, … -0.846269]","[0.39851, 8.586053, … -0.187713]"
1126087803073092178,2024-03-31,2024-05-31,207,0,3,1,"[3, 6, … 171]","[0.474452, -0.394806, … 1.770074]","[-291.889038, -50.377548, … -157.854645]"
1126087815958070773,2024-06-30,2024-08-31,375,0,2,0,"[3, 6, … 170]","[0.616849, -0.37807, … 0.755987]","[30.319223, -136.146622, … -153.045563]"
1126087854613040077,2024-03-31,2024-05-31,815,0,0,1,"[3, 6, … 171]","[-0.1204, -1.114465, … -0.136409]","[-31.038235, -132.840607, … -2.607502]"
1126087884678012428,2024-07-31,2024-09-30,280,0,3,0,"[3, 6, … 170]","[-0.177927, 0.224435, … -0.420353]","[71.578674, -114.807129, … -118.553825]"


## Tabular Collate Fn

In [3]:
# Read the docsting for details

dataset = TabularDataset(
    path=train_path,
    read_columns=None,
    shuffle_files=True,
    shuffle_pq=True,
    hidden_state_column="seq_hidden_state"
)

collate_fn = TabularCollateFn(
    target_column="target_attr_1",
    is_regression=False
)

dataloader = torch.utils.data.DataLoader(
    dataset=dataset,
    batch_size=256,
    num_workers=3,
    collate_fn=collate_fn
)

example_batch = next(iter(dataloader))

display(example_batch.keys())
display("dict_key: type", {key: type(val) for key, val in example_batch.items()})

dict_keys(['tab_features', 'epk_id', 'month_part', 'report_month', 'bucket_num', 'targets', 'target_attr_2', 'target_attr_3'])

'dict_key: type'

{'tab_features': fmlib.data.tabular_batch.TabularBatch,
 'epk_id': list,
 'month_part': list,
 'report_month': list,
 'bucket_num': list,
 'targets': torch.Tensor,
 'target_attr_2': list,
 'target_attr_3': list}